# 🤖 Notebook: Criando Agentes Acadêmicos Inteligentes com Azure AI

## O que foi aprendido neste notebook:

1. ✅ Como funciona agentes de IA e como funcionam
2. ✅ Como criar agentes especializados (Agente acadêmico)
3. ✅ Como usar "ferramentas" (tools) 
4. ✅ Como criar um orquestrador que gerencia múltiplos agentes
5. ✅ Programação assíncrona (async/await) de forma simples

---

## Objetivos de Aprendizado

- Estudar o conceito de agentes especializados
- Criar agentes com papéis, comportamentos e competências específicas
- Adicionar ferramentas usadas pelos agentes
- Orquestrar múltiplos agentes para trabalharem juntos

---

## Tecnologias usadas:

- **Microsoft Agent Framework**: Framework para criar e gerenciar agentes
- **Azure OpenAI**: Os modelos de IA que alimentam os agentes
- **Programação Assíncrona**: Para executar múltiplos agentes eficientemente

---

**Instalar dependências:**
```bash
pip install azure-ai-projects azure-identity agent-framework python-dotenv
```

---

In [ ]:
import sys
print(sys.executable)

c:\Users\Microsoft Windows\agente_ia_academica\.venv\Scripts\python.exe


In [6]:
import sys
import subprocess

subprocess.check_call([sys.executable, "-m", "pip", "install", "pypdf"])

0

In [7]:
from pypdf import PdfReader
print("Funcionou!")

Funcionou!


In [8]:
pip install azure-ai-projects azure-identity agent-framework python-dotenv


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
import os
import asyncio
from agent_framework import ChatAgent
from agent_framework.azure import AzureAIAgentClient
from azure.identity.aio import AzureCliCredential
from azure.ai.projects.aio import AIProjectClient
from agent_framework import ChatAgent
from agent_framework.azure import AzureOpenAIChatClient
from dotenv import load_dotenv
import re
import numpy as np
from pypdf import PdfReader
from openai import AsyncAzureOpenAI


load_dotenv()

endpoint_model = os.getenv("AZURE_MODEL_ENDPOINT") # Endpoint do modelo no Azure AI Foundry
deployment = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME") #Nome do deploy do modelo no Azure AI Foundry
model_key = os.getenv("AZURE_AI_MODEL_API_KEY") #Chave de API do modelo no Azure AI Foundry
embedding_deployment = os.getenv("AZURE_EMBEDDING_DEPLOYMENT")


In [24]:
import sys
import subprocess

subprocess.run([sys.executable, "-m", "pip", "show", "openpyxl"])


CompletedProcess(args=['c:\\Users\\Microsoft Windows\\agente_ia_academica\\.venv\\Scripts\\python.exe', '-m', 'pip', 'show', 'openpyxl'], returncode=0)

In [25]:
import os
from openpyxl import Workbook, load_workbook


def salvar_indexacao_planilha(palavras_chave, conceitos, termos, arquivo="indexacao.xlsx"):
    """
    Salva os resultados de indexação em uma planilha Excel.
    """

    print("\n=== TOOL EXECUTADA ===")

    # 🔎 mostra diretório atual
    print("Diretório atual:", os.getcwd())
    print("Arquivo será salvo em:", os.path.abspath(arquivo))

    # 🔎 mostra dados recebidos
    print("Palavras-chave:", palavras_chave)
    print("Conceitos:", conceitos)
    print("Termos:", termos)

    # 🔧 garante que tudo seja lista
    if isinstance(palavras_chave, str):
        palavras_chave = [palavras_chave]

    if isinstance(conceitos, str):
        conceitos = [conceitos]

    if isinstance(termos, str):
        termos = [termos]

    # 📂 cria ou abre planilha
    if os.path.exists(arquivo):
        wb = load_workbook(arquivo)
        ws = wb.active
    else:
        wb = Workbook()
        ws = wb.active
        ws.append(["Palavras-chave", "Conceitos principais", "Termos indexadores"])

    # 📊 escreve dados
    max_len = max(len(palavras_chave), len(conceitos), len(termos))

    for i in range(max_len):
        pk = palavras_chave[i] if i < len(palavras_chave) else ""
        c = conceitos[i] if i < len(conceitos) else ""
        t = termos[i] if i < len(termos) else ""
        ws.append([pk, c, t])

    # 💾 salva
    wb.save(arquivo)

    print("Arquivo salvo com sucesso!")

    return f"Dados salvos em {os.path.abspath(arquivo)}"

In [26]:
salvar_indexacao_planilha(
    ["indexação", "documentos audiovisuais"],
    ["representação do conhecimento"],
    ["polissemia da imagem"]
)


=== TOOL EXECUTADA ===
Diretório atual: c:\Users\Microsoft Windows\agente_ia_academica
Arquivo será salvo em: c:\Users\Microsoft Windows\agente_ia_academica\indexacao.xlsx
Palavras-chave: ['indexação', 'documentos audiovisuais']
Conceitos: ['representação do conhecimento']
Termos: ['polissemia da imagem']
Arquivo salvo com sucesso!


'Dados salvos em c:\\Users\\Microsoft Windows\\agente_ia_academica\\indexacao.xlsx'

In [27]:
from azure.identity.aio import AzureCliCredential

async def criar_agentes():

    chat_client = AzureOpenAIChatClient(
        endpoint=endpoint_model,
        api_key=model_key,
        deployment_name=deployment,
        api_version="2024-10-21",
    )

    # ========================================
    # AGENTE 1 — EXTRATOR
    # ========================================
    extrator_agent = ChatAgent(
        chat_client=chat_client,
        name="extrator_conceitos",
        instructions=(
            "Você é especialista em organização do conhecimento.\n\n"
            "Analise o texto e extraia:\n"
            "- palavras-chave\n"
            "- conceitos principais\n"
            "- termos indexadores\n\n"
            "Quando terminar, chame a ferramenta salvar_indexacao_planilha "
            "com os seguintes parâmetros:\n"
            "palavras_chave: lista de strings\n"
            "conceitos: lista de strings\n"
            "termos: lista de strings\n\n"
            "Nunca responda em texto simples."
        ),
        tools=[salvar_indexacao_planilha]
    )

    # ========================================
    # AGENTE 2 — REVISOR
    # ========================================
    revisor_agent = ChatAgent(
        chat_client=chat_client,
        name="revisor_academico",
        instructions=(
            "Você é um assistente especialista em revisão acadêmica.\n"
            "Corrija erros gramaticais, melhore clareza e mantenha rigor científico."
        ),
    )

    return extrator_agent, revisor_agent

In [28]:
async def rodar_agente():

    extrator_agent, revisor_agent = await criar_agentes()

    texto_usuario = """
Os instrumentos terminológicos desenvolvidos para a informação textual não podem ser
aplicados diretamente às imagens em movimento, uma vez que são concebidos para documentos
que apresentam uma estrutura linguística mais controlada, constituída por palavras, frases
e regras gramaticais. Esses instrumentos geralmente utilizam métodos como a identificação
de palavras-chave, conceitos principais e termos indexadores, recursos que não capturam
adequadamente as particularidades dos documentos audiovisuais. Ademais, as imagens em
movimento passam por um processo de transcodificação, isto é, a conversão do signo
imagético para um código textual, o que acarreta desafios inerentes a toda tradução, como
perda de precisão, seleção da informação e possibilidade de erro (Lima, 2019). Soma-se a
isso a polissemia intrínseca da imagem e da imagem em movimento, característica que
envolve múltiplos sentidos e que se relaciona diretamente com os processos de conotação e
denotação durante a análise documental. Portanto, devido à complexidade e à multiplicidade
de significados presentes nas imagens em movimento, torna-se necessário o desenvolvimento
de instrumentos terminológicos específicos, capazes de representar de forma adequada os
conteúdos audiovisuais e superar as limitações dos vocabulares controlados tradicionais
empregados para documentos textuais (Domínguez-Delgado; López-Hernández, 2017; Lima,
2016).
"""

    # ========================================
    # ETAPA 1 — EXTRAÇÃO CONCEITUAL
    # ========================================
    extracao = await extrator_agent.run(texto_usuario)

    # alguns frameworks retornam objeto
    if hasattr(extracao, "content"):
        extracao = extracao.content

    # ========================================
    # ETAPA 2 — REVISÃO ACADÊMICA
    # ========================================
    prompt_revisao = f"""
TEXTO ORIGINAL:
{texto_usuario}

ANÁLISE CONCEITUAL EXTRAÍDA:
{extracao}

TAREFA:
Corrija erros gramaticais, melhore a coesão e mantenha rigor acadêmico.
"""

    revisao = await revisor_agent.run(prompt_revisao)

    if hasattr(revisao, "content"):
        revisao = revisao.content

    # ========================================
    # RESULTADOS
    # ========================================
    print("\n==============================")
    print("EXTRAÇÃO CONCEITUAL")
    print("==============================\n")
    print(extracao)

    print("\n==============================")
    print("REVISÃO ACADÊMICA")
    print("==============================\n")
    print(revisao)

In [30]:
await rodar_agente()


=== TOOL EXECUTADA ===
Diretório atual: c:\Users\Microsoft Windows\agente_ia_academica
Arquivo será salvo em: c:\Users\Microsoft Windows\agente_ia_academica\indexacao.xlsx
Palavras-chave: instrumentos terminológicos, informação textual, imagens em movimento, documentos audiovisuais, transcodificação, polissemia, conotação, denotação, vocabulários controlados
Conceitos: instrumentos terminológicos para informação textual, transcodificação de imagem para código textual, polissemia da imagem em movimento, conotação e denotação na análise documental, limitações dos vocabulários controlados tradicionais, desenvolvimento de instrumentos específicos para conteúdos audiovisuais
Termos: informação textual, imagens em movimento, estrutura linguística, palavras-chave, conceitos principais, termos indexadores, transcodificação, signo imagético, código textual, perda de precisão, seleção da informação, possibilidade de erro, polissemia, múltiplos sentidos, conotação, denotação, análise documental,

## 🎭 Passo 7: Criando um Orquestrador

Agora vamos para o **nível avançado**! 🚀

### 🤔 O que é um Orquestrador?

Um **orquestrador** é como um **gerente inteligente** que:
- Recebe uma pergunta do usuário
- Decide qual especialista (agente) deve responder
- Direciona a pergunta para o agente certo
- Retorna a resposta

**Analogia do mundo real:**
Imagine uma central telefônica de uma empresa:
- 🧮 Você liga perguntando sobre matemática → transfere para o departamento de contabilidade
- ✈️ Você liga perguntando sobre viagens → transfere para o departamento de turismo
- 💾 Você pede para salvar algo → usa a ferramenta de arquivos

### 🎯 Como funciona:

```
Usuário → Orquestrador → Decide qual agente usar → Agente especializado → Resposta
```

### 🔧 Agentes como Tools:

A parte INCRÍVEL: podemos transformar agentes em ferramentas usando `.as_tool()`!

```python
tools=[
    math_agent.as_tool(),      # ← Agente de matemática como ferramenta
    travel_agent.as_tool(),    # ← Agente de viagens como ferramenta
    salvar_texto_em_arquivo    # ← Função normal como ferramenta
]
```

O orquestrador vai **decidir sozinho** qual usar baseado na pergunta!

**Execute a célula abaixo:**

In [32]:
from azure.identity.aio import AzureCliCredential

async def criar_orquestrador():
    """
    Cria um orquestrador que gerencia os agentes:
    - extrator_conceitos
    - revisor_academico
    """

    credential = AzureCliCredential()

    chat_client = AzureOpenAIChatClient(
        endpoint=endpoint_model,
        api_key=model_key,
        deployment_name=deployment,
        api_version="2024-10-21",
    )

    # ========================================
    # AGENTE EXTRATOR 🔎
    # ========================================
    extrator_agent = ChatAgent(
        chat_client=chat_client,
        name="extrator_conceitos",
        instructions=(
            "Você é especialista em organização do conhecimento.\n\n"
            "Analise o texto e extraia:\n"
            "- palavras-chave\n"
            "- conceitos principais\n"
            "- termos indexadores\n\n"
            "Depois utilize a ferramenta salvar_indexacao_planilha "
            "para salvar os resultados."
        ),
        tools=[salvar_indexacao_planilha]
    )

    # ========================================
    # AGENTE REVISOR ✍️
    # ========================================
    revisor_agent = ChatAgent(
        chat_client=chat_client,
        name="revisor_academico",
        instructions=(
            "Você é um assistente especialista em revisão acadêmica.\n"
            "Corrija erros gramaticais, melhore clareza e mantenha rigor científico."
        ),
    )

    # ========================================
    # ORQUESTRADOR 🎭
    # ========================================
    orchestrator = ChatAgent(
        chat_client=chat_client,
        name="orchestrator-agent",
        instructions=(
            "Você é um orquestrador inteligente que decide qual agente usar.\n\n"

            "🔎 Use 'extrator_conceitos' quando o usuário pedir:\n"
            "- extração de palavras-chave\n"
            "- identificação de conceitos\n"
            "- indexação de textos\n"
            "- análise conceitual\n"
            "- organização do conhecimento\n\n"

            "✍️ Use 'revisor_academico' quando o usuário pedir:\n"
            "- revisão de texto\n"
            "- correção gramatical\n"
            "- melhoria de clareza\n"
            "- revisão acadêmica\n\n"

            "Explique em uma frase qual agente foi usado antes da resposta.\n"
            "Sempre utilize o resultado do agente escolhido."
        ),

        # transformando agentes em tools
        tools=[
            extrator_agent.as_tool(),
            revisor_agent.as_tool()
        ],
    )

    return orchestrator, credential

## 🎬 Passo 8: Testando o Orquestrador

Vamos fazer um teste completo! Vamos pedir:
1. Um roteiro de viagem para a Bahia
2. Para salvar em um arquivo .txt

### 🎯 O que vai acontecer:

1. **Orquestrador** recebe a pergunta
2. **Orquestrador** identifica que é sobre viagem
3. **Orquestrador** chama o `travel_agent` (agente pirata!)
4. **travel_agent** cria o roteiro
5. **Orquestrador** identifica que precisa salvar
6. **Orquestrador** chama a tool `salvar_texto_em_arquivo`
7. Arquivo é salvo na sua pasta!

**Tudo isso automaticamente!** 🤯

### 💡 Experimente Depois:

Mude a pergunta para testar diferentes cenários:
- `"Quanto é 150 * 78?"` → Vai usar o math_agent
- `"Roteiro para Rio de Janeiro"` → Vai usar o travel_agent
- `"Qual é a capital da França?"` → Orquestrador responde direto

**Execute a célula abaixo:**

In [33]:
async def rodar_orquestrador():

    orchestrator, credential = await criar_orquestrador()

    pergunta = """
Extraia conceitos candidatos deste texto:

Os instrumentos terminológicos desenvolvidos para a informação textual não podem ser
aplicados diretamente às imagens em movimento, uma vez que são concebidos para documentos
que apresentam uma estrutura linguística mais controlada, constituída por palavras, frases
e regras gramaticais. Esses instrumentos geralmente utilizam métodos como a identificação
de palavras-chave, conceitos principais e termos indexadores, recursos que não capturam
adequadamente as particularidades dos documentos audiovisuais. Ademais, as imagens em
movimento passam por um processo de transcodificação, isto é, a conversão do signo
imagético para um código textual, o que acarreta desafios inerentes a toda tradução, como
perda de precisão, seleção da informação e possibilidade de erro (Lima, 2019). Soma-se a
isso a polissemia intrínseca da imagem e da imagem em movimento, característica que
envolve múltiplos sentidos e que se relaciona diretamente com os processos de conotação e
denotação durante a análise documental. Portanto, devido à complexidade e à multiplicidade
de significados presentes nas imagens em movimento, torna-se necessário o desenvolvimento
de instrumentos terminológicos específicos, capazes de representar de forma adequada os
conteúdos audiovisuais e superar as limitações dos vocabulares controlados tradicionais
empregados para documentos textuais (Domínguez-Delgado; López-Hernández, 2017; Lima,
2016).
"""

    resposta = await orchestrator.run(pergunta)

    if hasattr(resposta, "content"):
        resposta = resposta.content

    print("\n==============================")
    print("RESPOSTA DO ORQUESTRADOR")
    print("==============================\n")

    print(resposta)

In [34]:
await rodar_orquestrador()


=== TOOL EXECUTADA ===
Diretório atual: c:\Users\Microsoft Windows\agente_ia_academica
Arquivo será salvo em: c:\Users\Microsoft Windows\agente_ia_academica\indexacao.xlsx
Palavras-chave: extração, conceitos, texto, análise, organização do conhecimento
Conceitos: extração de conceitos, análise de texto, organização do conhecimento, conceitos candidatos
Termos: conceitos, extração, análise, texto, organização, conhecimento, candidatos

RESPOSTA DO ORQUESTRADOR

Usei o agente de extração de conceitos para identificar os conceitos candidatos do texto. Os conceitos extraídos são:

- Instrumentos terminológicos
- Informação textual
- Imagens em movimento
- Estrutura linguística controlada
- Palavras-chave
- Conceitos principais
- Termos indexadores
- Documentos audiovisuais
- Transcodificação
- Signo imagético
- Código textual
- Tradução (perda de precisão, seleção da informação, possibilidade de erro)
- Polissemia da imagem
- Conotação
- Denotação
- Análise documental
- Vocabularios contr

## 📚 Próximos Passos - Workflows

### 🎓 Indo Mais Fundo

O **Microsoft Agent Framework** recomenda a utilização de **workflows** para orquestração mais complexa e conexão entre agentes. 

**Workflows** são como "roteiros" pré-definidos onde você pode:
- 🔄 Criar fluxos de trabalho com múltiplas etapas
- ⚡ Executar agentes em sequência ou paralelo
- 🔀 Criar ramificações condicionais (if/else)
- 📊 Gerenciar estados complexos

### 📖 Documentação Oficial

Este tipo de código é mais avançado e pode ser aprofundado na documentação oficial:

🔗 [Microsoft Agent Framework - GitHub](https://github.com/microsoft/agent-framework/tree/main)

**Recomendado para:**
- Sistemas com múltiplos agentes interconectados
- Processos de negócio complexos
- Automações sofisticadas

---

## 🎉 Parabéns! Você completou o Notebook 2!

### ✅ O que você aprendeu:

1. **Conceito de Agentes**: O que são e como funcionam
2. **Agentes Especializados**: Como criar agentes com personalidades únicas
3. **Tools (Ferramentas)**: Como dar superpoderes aos agentes
4. **Orquestração**: Como gerenciar múltiplos agentes trabalhando juntos
5. **Programação Assíncrona**: Como usar `async`/`await` de forma prática
6. **Arquitetura de Sistemas**: Como estruturar sistemas de IA complexos

---

## 💡 Ideias para Praticar

### 1. Crie Seus Próprios Agentes 🎨

Experimente criar agentes personalizados, por exemplo:
- 📚 Agente Educador: Explica conceitos de forma simples
- 🎵 Agente Musical: Recomenda músicas e playlists
- 🏋️ Agente Fitness: Cria treinos personalizados
- 👨‍💻 Agente Programador: Ajuda com código

### 2. Crie Suas Próprias Tools 🛠️

Ideias de ferramentas úteis:
- 📊 Gerar gráficos com matplotlib
- 📧 Enviar emails
- 🌐 Buscar informações na internet
- 📅 Gerenciar calendário
- 💵 Fazer conversões de moeda

### 3. Orquestradores Mais Complexos 🎭

- Crie um orquestrador com 5+ agentes
- Adicione lógica de prioridade
- Implemente fallbacks (planos B)

---

## 🔑 Conceitos-Chave para Lembrar

### 📌 Agentes
```python
ChatAgent(
    name="nome-unico",
    instructions="Seu manual de trabalho detalhado",
    tools=[lista_de_ferramentas]
)
```

### 📌 Tools
- Sempre use **type hints** (`parametro: tipo`)
- Sempre adicione **docstrings**
- Funções devem fazer **uma coisa bem feita**

### 📌 Orquestração
- Transforme agentes em tools com `.as_tool()`
- Dê instruções claras sobre **quando usar cada ferramenta**
- Teste diferentes cenários

---

## 🆘 Troubleshooting

### Erro: "Tool not found"
✓ Verifique se adicionou a tool na lista `tools=[]`
✓ Certifique-se de que a função tem type hints

### Erro: "Agent não usa a tool"
✓ Revise as `instructions` - seja mais específica
✓ Mencione explicitamente quando usar a tool
✓ Teste com perguntas mais diretas

### Arquivo não foi salvo
✓ Verifique se tem permissões na pasta
✓ Olhe na pasta do notebook (mesma pasta que o .ipynb)
✓ Confira se o caminho do arquivo está correto

---

## 📖 Recursos Adicionais

### Documentação Oficial
- [Microsoft Agent Framework](https://github.com/microsoft/agent-framework)
- [Azure AI Foundry](https://learn.microsoft.com/azure/ai-studio/)
- [Azure OpenAI Service](https://learn.microsoft.com/azure/ai-services/openai/)

### Conceitos Avançados
- [Prompt Engineering](https://learn.microsoft.com/azure/ai-services/openai/concepts/prompt-engineering)
- [RAG (Retrieval Augmented Generation)](https://learn.microsoft.com/azure/ai-services/openai/concepts/use-your-data)
- [Function Calling](https://learn.microsoft.com/azure/ai-services/openai/how-to/function-calling)

### Tutoriais
- [Build AI Agents](https://learn.microsoft.com/azure/ai-studio/tutorials/deploy-chat-web-app)
- [Azure AI Samples](https://github.com/Azure-Samples/azure-ai-samples)

---

## 🌟 Próximos Desafios

Agora que você domina os fundamentos, explore:

1. **RAG (Retrieval Augmented Generation)**
   - Ensine seus agentes a buscar informações em documentos
   
2. **Multimodal Agents**
   - Agentes que trabalham com imagens, áudio e texto
   
3. **Persistent Memory**
   - Agentes que lembram conversas anteriores
   
4. **Production Deployment**
   - Deploy seus agentes em produção com Azure App Service

---

## 🤝 Compartilhe seu Aprendizado

Criou algo legal? Compartilhe!
- Faça um fork do repositório
- Adicione seus agentes personalizados
- Crie um Pull Request
- Inspire outras pessoas! 💜

---

## 🎓 Conclusão

Você agora tem as habilidades fundamentais para:
- ✨ Criar aplicações de IA inteligentes
- 🚀 Desenvolver sistemas multi-agentes
- 🛠️ Integrar ferramentas personalizadas
- 🎯 Resolver problemas reais com IA

**O céu é o limite! Continue explorando e criando! 🌟**

---

## 💬 Feedback

Tem sugestões para melhorar este material?
- Abra uma issue no GitHub
- Compartilhe suas ideias
- Contribua com novos exemplos

**Obrigada por aprender conosco! 🙏✨**

---

<div style="text-align: center; font-size: 20px; margin-top: 30px;">
🎉 <strong>PARABÉNS!</strong> 🎉<br>
Você completou o curso de Fundamentos de IA com Azure!<br>
<strong>Continue aprendendo, continue criando! 🚀💜</strong>
</div>